In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pickle

In [3]:
ratings_df = pd.read_csv('../data/raw/ratings.csv')
books_df = pd.read_csv('../data/raw/books.csv')

In [4]:
def cleanData(df, text_columns, genre_columns):
  clean_books_df = df.copy(deep=True)
  clean_books_df['group_text'] = clean_books_df[text_columns].astype(str).agg(" ".join, axis=1)  

  clean_books_df = clean_books_df[["book_id", genre_columns, "group_text"]]

  return clean_books_df

clean_books_df = cleanData(
    books_df, 
    ['description', 'authors'], 
    'genres'
)

clean_books_df = clean_books_df.sort_values('book_id').reset_index(drop=True)
clean_books_df.to_csv('../data/processed/clean_books_df.csv', index=False)

In [5]:
def preprocessText(text):
  lowered_text = text.lower()

  remove_punctuation = re.sub(r'[^\w\s]', '', lowered_text)
  remove_numbers = re.sub(r'[^a-zA-Z\s]', ' ', remove_punctuation)
  clean_text = re.sub(r'\s+', ' ', remove_numbers).strip()

  return clean_text

In [6]:
def build_vectors(clean_books_text):
  vectorizer = TfidfVectorizer(
     stop_words='english', # ignore random english words that don't add meaning
     min_df = 2, # ignore words that appear in less than 2 books
     max_df = 0.9, # ignores words that appear in 90%+ books
     ngram_range=(1,2) # use unigram and bigram
  )
  result =  vectorizer.fit_transform(clean_books_text) # contains rows: books; columns: words
  return vectorizer, result

In [7]:
clean_books_group_text = clean_books_df['group_text'].apply(preprocessText)
clean_books_genre = clean_books_df['genres'].apply(preprocessText)

print("Processed Text: ")
print(clean_books_group_text.head())

print("Processed Genres: ")
print("", clean_books_genre.head())

Processed Text: 
0    winning means fame and fortunelosing means cer...
1    harry potters life is miserable his parents ar...
2    about three things i was absolutely positive f...
3    the unforgettable novel of a childhood in a sl...
4    alternate cover edition isbn isbn the great ga...
Name: group_text, dtype: object
Processed Genres: 
 0    youngadult fiction fantasy sciencefiction romance
1                  fantasy fiction youngadult classics
2        youngadult fantasy romance fiction paranormal
3        classics fiction historicalfiction youngadult
4           classics fiction historicalfiction romance
Name: genres, dtype: object


In [8]:
group_text_vector, group_text_result = build_vectors(clean_books_group_text)
genre_vector, genre_result = build_vectors(clean_books_genre)

In [9]:
ratings_matrix = ratings_df.pivot(index='book_id', columns='user_id', values='rating').fillna(0)

user_rating_similarities = cosine_similarity(ratings_matrix)

ratings_matrix.head()


user_id,1,2,3,4,5,6,7,8,9,10,...,53415,53416,53417,53418,53419,53420,53421,53422,53423,53424
book_id,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,4.0,5.0,4.0,4.0,4.0,4.0,4.0,4.0
2,0.0,5.0,0.0,5.0,0.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,5.0,5.0,5.0,5.0,5.0,5.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,...,0.0,0.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,4.0
4,5.0,0.0,3.0,4.0,0.0,0.0,0.0,3.0,0.0,5.0,...,0.0,0.0,0.0,0.0,3.0,0.0,5.0,0.0,5.0,5.0
5,0.0,5.0,0.0,4.0,0.0,0.0,3.0,3.0,5.0,5.0,...,0.0,0.0,0.0,0.0,3.0,2.0,4.0,0.0,0.0,0.0


In [10]:
liked_book_id = 68 # perks of being a wallflower

def bookRecommendation(liked_book_id):

  book_id_list = clean_books_df['book_id'].tolist() # convert book_id column into a list

  book_desc_vectors = group_text_result
  genre_vectors= genre_result

  liked_index = book_id_list.index(liked_book_id) #finds the row that the liked_book_id is in

  ratings_similarities = user_rating_similarities[liked_index]
  text_similarities = cosine_similarity(book_desc_vectors[liked_index], book_desc_vectors).flatten()
  genre_similarities = cosine_similarity(genre_vectors[liked_index], genre_vectors).flatten()

  similarities = 0.5 * ratings_similarities + 0.3 * text_similarities + 0.2 * genre_similarities
  similarities[liked_index] = -1  # sets the book itself to index -1 so it won't be recommended

  top_indices = similarities.argsort()[-5:][::-1] # return top 5 indices

  recommended_books = []
  
  for i in top_indices:
    recommended_books.append(book_id_list[i])

  print("Recommended book IDs:", recommended_books)

bookRecommendation(liked_book_id)

Recommended book IDs: [74, 6, 369, 564, 698]


In [ ]:
artifacts = {
  "group_text_result": group_text_result,
  "genre_result": genre_result,
  "user_rating_similarities": user_rating_similarities,
  "book_id_list": clean_books_df['book_id'].tolist(),
}

with open('../data/processed/artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)